In [ ]:
## for chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
## convert text to doc
from langchain_community.document_loaders import TextLoader,DirectoryLoader
##utility
import numpy as np
import os
from typing import List
## to convert text into vector embeddings
from langchain_huggingface import embeddings
## to store vectors
import chromadb


os.makedirs("data",exist_ok=True)

text={
"data/py.txt":"""Python is one of the most popular programming languages in the world. 
It is widely used in web development, machine learning, artificial intelligence, 
automation, and data science because of its simple syntax and powerful libraries.""",

"data/emb.txt":"""Embeddings are numerical vector representations of text. 
They help machines understand semantic meaning by converting words, sentences, 
or documents into high-dimensional vectors that capture relationships between texts.""",


"data/cos.txt":"""Cosine similarity is a mathematical technique used to measure how similar two vectors are. 
Instead of comparing exact words, it compares the angle between vectors, making it useful 
for semantic search, recommendation systems, and Retrieval-Augmented Generation (RAG) applications.""",

"data/rag.txt":"""In a RAG pipeline, documents are converted into embeddings and stored in a vector database. 
When a user asks a question, the query is also converted into an embedding, and cosine similarity 
is used to retrieve the most relevant documents before generating the final answer."""
}

for file_path,content in text.items():
    with open(file_path,'w',encoding="utf-8") as f:
        f.write(content)


In [ ]:
from langchain_chroma import Chroma
dirs = DirectoryLoader("data", glob="*.txt",loader_cls=TextLoader)
docs = dirs.load()
# print(docs)

text_splitter = RecursiveCharacterTextSplitter(chunk_size=100,chunk_overlap=20,separators=["\n\n","\n"," ",""])
chunks = text_splitter.split_documents(docs)
print(chunks)

# per_dir="./chroma_db"

chromaClient = chromadb.Client()
store = Chroma(
    collection_name="bhasrdsa"
)
# store = chromaClient.create_collection(name="bhasra")

## convert text to vector embeddings
# embed = embeddin+gs.HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")



In [ ]:
# store.add(
#     documents=[doc.page_content for i,doc in enumerate(chunks)],
#     ids=[str(i) for i, _ in enumerate(chunks)]
# )
store.add_documents(documents=chunks)
# store.get()
# print(store.count())
# print(store.get())

In [ ]:
qu= "cosine"
# sim = store.query(query_texts=[query])
sim = store.similarity_search(query=qu)
# print(store.search(query))
# advsearch = store.get(where_document={"$contains":query})
print(sim)
# print(advsearch)

In [ ]:
# from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
llm = ChatOllama(
    model="phi3"
)
test = llm.invoke("what is LLM")
test

In [ ]:
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

ret = store.as_retriever(
    kwargs={"k":3}
)
print(ret)

In [ ]:
sys_prompt = """ you are an assistant for question-answering tasks. use the fallowing pieces of retrieved context to answer
the question. if you don't know the answer, just say that you don't know use three sentences maximum and 
keep the answers concise context:{context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system",sys_prompt),
    ("human","{input}")
])

chain = create_stuff_documents_chain(llm,prompt)
# chain
rag_chain = create_retrieval_chain(ret,chain)
rag_chain.invoke({"input":"What is Python?"})
